In [0]:
%sql
CREATE OR REPLACE TABLE dev_credit_risk.gold.fact_parcela_credito
USING DELTA

AS

SELECT

    p.id_parcela,
    p.id_contrato,

    c.id_cliente,
    c.id_produto,
    c.id_score,

    p.numero_parcela,

    p.dt_competencia,
    p.dt_vencimento,

    pg.dt_pagamento,

    p.valor_parcela,

    pg.valor_pago,

    pg.dias_atraso,

    CASE
        WHEN pg.dt_pagamento IS NOT NULL THEN 1
        ELSE 0
    END AS flag_pagamento,

    CASE
        WHEN pg.dt_pagamento IS NOT NULL
        AND pg.dias_atraso > 0
        THEN 1
        ELSE 0
    END AS flag_pagamento_atrasado,

    CASE
        WHEN pg.dt_pagamento IS NOT NULL
         AND pg.dias_atraso >= 90
        THEN 1
        ELSE 0
    END AS flag_pagamento_default_90,

    CASE
        WHEN pg.dt_pagamento IS NULL
        THEN p.valor_parcela
        ELSE CAST(0 AS DECIMAL(18,2))
    END AS valor_em_aberto,

    current_timestamp() AS dtCarga

FROM dev_credit_risk.silver.fact_parcela p

INNER JOIN dev_credit_risk.silver.fact_contrato c
    ON p.id_contrato = c.id_contrato

LEFT JOIN dev_credit_risk.silver.fact_pagamento pg
    ON p.id_parcela = pg.id_parcela;

In [0]:
%sql
select * from dev_credit_risk.gold.fact_parcela_credito